# Embedding Model Fine-Tuning
This notebook demonstrates how to fine-tune an open-source embedding model on your specific domain data. It uses the concept of **Knowledge Distillation** via Scored Triplets: an LLM generates questions, an existing embedding model finds 'hard negatives' (wrong but similar answers), and a powerful Cross-Encoder scores them. The smaller embedding model learns to mimic these scores using MarginMSELoss.

## Cell 1: Setup & Imports
We import the necessary libraries. We use `openai` to interact with our LLM for generating questions, `pandas` for handling our data, `pickle` to save our temporary dataset, and `sentence-transformers` for the actual fine-tuning process. We also import `difflib` for data cleaning.

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import re
import random
import difflib
from openai import OpenAI, AsyncOpenAI
import asyncio
import nest_asyncio
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

## Cell 2: API Clients & Models
We connect to our local vLLM API to access our LLM, Embedding, and Reranker (Cross-Encoder) models.

In [ ]:
API_KEY = "dummy-key"

THINKING_LLM_URL = "http://localhost:8000/v1"
EMBEDDING_MODEL_URL = "http://localhost:8002/v1"
RERANKER_MODEL_URL = "http://localhost:8003/v1"

# Create API clients for dataset generation
llm_client = OpenAI(base_url=THINKING_LLM_URL, api_key=API_KEY)
async_llm_client = AsyncOpenAI(base_url=THINKING_LLM_URL, api_key=API_KEY)
embedding_client = OpenAI(base_url=EMBEDDING_MODEL_URL, api_key=API_KEY)
reranker_client = OpenAI(base_url=RERANKER_MODEL_URL, api_key=API_KEY)

# Define models hosted on vLLM
THINKING_LLM = "meta-llama/Llama-3-70b-Instruct"
EMBEDDING_MODEL = "mixedbread-ai/mxbai-embed-large-v1"
RERANKER_MODEL = "mixedbread-ai/mxbai-rerank-large-v1"

def check_connections():
    try:
        print("LLM Connection:", llm_client.models.list().data[0].id)
    except:
        print("LLM Connection: Offline (Mocking generation for training)")
        
check_connections()

## Cell 3: Loading Domain Data
We load the exact same data as our RAG pipeline (Documents, Lookup Data, and Document Metadata).

In [ ]:
# 1. Main Documents
docs_data = [
    {"file_name": f"doc_{i}.txt", "content": f"## Intro\n\nHere is detailed info about the AWS protocol and EC2 instances. Section {i} covers routing and subnets. This is repeated text. This is repeated text. %%% ___"} 
    for i in range(1, 101) # Using 100 for fast dataset generation
]
docs_df = pd.DataFrame(docs_data)

# 2. Lookup Data
lookup_df = pd.DataFrame([
    {"acronym": "AWS", "definition": "Amazon Web Services"},
    {"acronym": "EC2", "definition": "Elastic Compute Cloud"}
])

# 3. Document Metadata
metadata_df = pd.DataFrame([
    {"file_name": "doc_1.txt", "category": "Networking", "description": "IP addresses, routers, and subnets."}
])

print(f"Loaded {len(docs_df)} documents for training.")

## Cell 4: Data Preprocessing & Chunking
**CRITICAL:** An embedding model must be trained on the exact same format of data it will see in production. Here we use the exact same cleaning, structural chunking, and fuzzy-match acronym augmentation from our main RAG pipeline notebook.

In [ ]:
def clean_text(text):
    text = re.sub(r'[%_]{2,}', '', text)
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    unique_sentences = []
    for s in sentences:
        if not unique_sentences or s != unique_sentences[-1]:
            unique_sentences.append(s)
    return '. '.join(unique_sentences) + '.'

def chunk_document(text, max_chars=1000):
    paragraphs = text.split('\n\n')
    chunks = []
    current_chunk = ""
    for p in paragraphs:
        if len(current_chunk) + len(p) < max_chars:
            current_chunk += p + "\n\n"
        else:
            if current_chunk.strip(): chunks.append(current_chunk.strip())
            current_chunk = p + "\n\n"
    if current_chunk.strip(): chunks.append(current_chunk.strip())
    return [c for c in chunks if len(re.sub(r'\W+', '', c)) > 20]

def augment_chunk(chunk_text, lookup_df):
    augmented_text = chunk_text
    valid_acronyms = lookup_df['acronym'].tolist()
    words = set(re.findall(r'\b[A-Za-z0-9_]+\b', chunk_text))
    for word in words:
        if len(word) < 2: continue
        matches = difflib.get_close_matches(word, valid_acronyms, n=1, cutoff=0.8)
        if matches:
            matched_acronym = matches[0]
            definition = lookup_df.loc[lookup_df['acronym'] == matched_acronym, 'definition'].values[0]
            pattern = r'\b' + re.escape(word) + r'\b'
            replacement = f"{matched_acronym} ({definition})"
            augmented_text = re.sub(pattern, replacement, augmented_text)
    return augmented_text

processed_chunks = []
for _, row in docs_df.iterrows():
    clean_txt = clean_text(row['content'])
    chunks = chunk_document(clean_txt)
    for c in chunks:
        processed_chunks.append(augment_chunk(c, lookup_df))
        
print(f"Processed into {len(processed_chunks)} cleaned and augmented chunks.")

## Cell 5: Synthetic Question Generation (Anchors)
To get enough data for fine-tuning (ideally 5k-20k triplets), we need multiple questions per chunk. We use an asynchronous LLM client (`AsyncOpenAI`) and `asyncio` to make up to 10 concurrent requests to the LLM. This drastically speeds up dataset generation. We ask the LLM to generate diverse questions (fact-based, conceptual, troubleshooting) for each chunk. The number of questions per chunk is easily configurable.

In [ ]:
# Configuration: Easy to change later based on how many chunks you have
QUESTIONS_PER_CHUNK = 3
CONCURRENCY_LIMIT = 10

# Required for running asyncio loops inside Jupyter Notebooks safely
nest_asyncio.apply()
semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

async def generate_questions_async(chunk_text, num_questions=QUESTIONS_PER_CHUNK):
    """Uses the LLM asynchronously to generate multiple diverse questions for a single chunk."""
    prompt = f"""Given the following text, write {num_questions} distinct, realistic search queries or questions that a user would type to find this exact information. 
Try to vary the types of questions (e.g., one fact-based, one conceptual, one troubleshooting).
Return ONLY the questions, one per line, without numbering or bullet points.

Text: {chunk_text}"""
    
    async with semaphore:
        try:
            response = await async_llm_client.chat.completions.create(
                model=THINKING_LLM,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7
            )
            content = response.choices[0].message.content.strip()
            # Clean up bullet points, numbers, and empty lines
            questions = [re.sub(r'^[-*0-9.\s]+', '', q).strip() for q in content.split('\n') if q.strip()]
            return questions[:num_questions]
        except Exception as e:
            # Fallback if API is offline
            return [f"What is fact {i} about this section?" for i in range(1, num_questions + 1)]

# Test it on one chunk
sample_chunk = processed_chunks[0]
sample_queries = await generate_questions_async(sample_chunk)
print("Chunk:\n", sample_chunk, "\n")
print(f"Generated {len(sample_queries)} Queries:")
for q in sample_queries:
    print("-", q)

## Cell 6: Hard Negative Mining & Scoring
For every single question generated, we find a 'Hard Negative' chunk (similar text, but wrong answer) using our existing embedding model. Then we score the (Question, Correct Chunk) and (Question, Wrong Chunk) using the Cross-Encoder.

In [ ]:
def get_embedding_mock(text):
    try:
        return embedding_client.embeddings.create(input=text, model=EMBEDDING_MODEL).data[0].embedding
    except:
        return np.random.rand(1024)

def get_ce_score_mock(query, doc):
    overlap = len(set(query.split()) & set(doc.split()))
    return min(0.99, 0.1 + (overlap * 0.1))

# 1. Pre-calculate all chunk embeddings to search for hard negatives
print("Pre-calculating embeddings for hard negative search...")
chunk_embeddings = [get_embedding_mock(c) for c in processed_chunks]

# Limit loop to 5 chunks for demo speed (would be all processed_chunks in production)
chunks_to_process = processed_chunks[:5]

# 2. Generate questions asynchronously for all chunks
print(f"\nAsynchronously generating questions ({QUESTIONS_PER_CHUNK} per chunk) for {len(chunks_to_process)} chunks...")
async def fetch_all_questions():
    tasks = [generate_questions_async(chunk, QUESTIONS_PER_CHUNK) for chunk in chunks_to_process]
    return await asyncio.gather(*tasks)

# Top-level await works natively in modern Jupyter environments
all_questions = await fetch_all_questions()

# 3. Build the Scored Triplet Dataset
dataset = []
print("Generating triplets and scoring hard negatives...")

for i, positive_chunk in enumerate(chunks_to_process): 
    questions = all_questions[i]
    
    for query in questions:
        # Hard Negative Search
        query_emb = get_embedding_mock(query)
        similarities = [np.dot(query_emb, ce) for ce in chunk_embeddings]
        sorted_indices = np.argsort(similarities)[::-1]
        
        # Ensure the negative isn't the positive chunk itself
        negative_idx = sorted_indices[0] if sorted_indices[0] != i else sorted_indices[1]
        negative_chunk = processed_chunks[negative_idx]
        
        # Cross-Encoder Scoring
        pos_score = get_ce_score_mock(query, positive_chunk)
        neg_score = get_ce_score_mock(query, negative_chunk)
        
        dataset.append({
            "query": query,
            "pos": positive_chunk,
            "neg": negative_chunk,
            "pos_score": pos_score,
            "neg_score": neg_score
        })

print(f"\nGenerated {len(dataset)} scored triplets.")

## Cell 7: Save Temporary Dataset
We save our generated training dataset to a local pickle file. In a real environment, this generation step might take hours, so caching it to disk ensures we don't lose our expensive LLM outputs.

In [ ]:
TMP_FILE = "/tmp/embedding_training_triplets.pkl"

with open(TMP_FILE, 'wb') as f:
    pickle.dump(dataset, f)
    
print(f"Saved dataset to {TMP_FILE}")

## Cell 8: Model Fine-Tuning (MarginMSELoss)
Finally, we use `sentence-transformers` to fine-tune the model. We use `MarginMSELoss`, which is explicitly designed for Knowledge Distillation. The model looks at the `(Query, Positive)` and `(Query, Negative)` inputs, and tries to adjust its weights so that the difference (margin) between its embedding distances exactly matches the difference between the Cross-Encoder scores!

In [ ]:
# 1. Load the dataset
with open(TMP_FILE, 'rb') as f:
    loaded_dataset = pickle.load(f)

# 2. Format for sentence-transformers
train_examples = []
for data in loaded_dataset:
    margin = data['pos_score'] - data['neg_score']
    example = InputExample(texts=[data['query'], data['pos'], data['neg']], label=float(margin))
    train_examples.append(example)

# Create DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)

# 3. Load a pre-trained embedding model locally
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading local model: {model_name}")
model = SentenceTransformer(model_name)

# 4. Define the Loss Function (Knowledge Distillation)
train_loss = losses.MarginMSELoss(model=model)

# 5. Execute Fine-Tuning
print("Starting fine-tuning...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=10,
    show_progress_bar=True
)

# 6. Save the newly fine-tuned model
OUTPUT_PATH = "/tmp/fine_tuned_domain_embedder"
model.save(OUTPUT_PATH)
print(f"Fine-tuning complete! Model saved to {OUTPUT_PATH}")